# agents_runtime Smoke Test

Quick notebook to validate the extracted `agents_runtime` package end-to-end.

In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv

# Find repo root and add agents_runtime/src to PYTHONPATH.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = None
for c in candidates:
    if (c / 'agents_runtime' / 'src' / 'agents_runtime').is_dir():
        repo_root = c
        break
if repo_root is None:
    raise RuntimeError('Could not find agents_runtime/src from current working directory')

pkg_src = repo_root / 'agents_runtime' / 'src'
if str(pkg_src) not in sys.path:
    sys.path.insert(0, str(pkg_src))

load_dotenv(repo_root / '.env')

from agents_runtime.config import RuntimeSettings
from agents_runtime.mcp import build_servers
from agents_runtime.runtime import RuntimeContext, stream_chat_request

print('Repo root:', repo_root)
print('Package src:', pkg_src)
from IPython.display import Markdown, display


Repo root: /home/preshen/Projects/ai_analytics
Package src: /home/preshen/Projects/ai_analytics/agents_runtime/src


In [2]:
DB_KEY = 'mssql_demo'

USER_PROMPT1 = (
    'Find the latest CreationDate in dbo.CompletedChecklist, then show daily completion counts '
    'for the 30 days ending on that latest date. Generate the most appropriate chart if applicable. '
    'Only generate one chart.'
)

USER_PROMPT2 = (
    'Can you give me a representation of how the latest questionare /checklist form woudl look for say this checklist Riegelwood Mill - Safety Observation'
)

USER_PROMPT = (
    'Which OCR checklist from what sites had the most negative incidents for the first month of 2025, '
    'use the following tables and more [dbo].[Checklist], [dbo].[ChecklistStructure], [dbo].[CompletedChecklist], '
    '[dbo].[CompletedChecklist_Answer], Try other related tables. i think it might be in a Entity-Attribute-Value (EAV) '
)


In [3]:
settings = RuntimeSettings.from_env()
servers = build_servers(timeout_seconds=settings.mcp_client_timeout_seconds)
ctx = RuntimeContext.create(settings, servers)

print('Model:', settings.model_general)
print('Chroma:', settings.chroma_dir)


Model: gpt-4.1-mini
Chroma: .chroma_ai_analytics


In [4]:
# Stream only final markdown output
from IPython.display import Markdown, display

async def _stream_markdown(prompt: str):
    text_buf = ''
    handle = display(Markdown('_Waiting for model output..._'), display_id=True)

    async with (
        servers.mssql_server,
        servers.chart_server,
        servers.pdf_reader_server,
        servers.word_reader_server,
        servers.excel_server,
    ):
        def on_text_delta(delta: str):
            nonlocal text_buf
            if not delta:
                return
            text_buf += delta
            handle.update(Markdown(text_buf))

        final = await stream_chat_request(
            ctx=ctx,
            user_prompt=prompt,
            db_key=DB_KEY,
            on_text_delta=on_text_delta,
        )

    if not text_buf.strip():
        text_buf = final
        handle.update(Markdown(text_buf))

    return final

final = await _stream_markdown(USER_PROMPT1)
final


[status] Working on the next part
[status] Analyzing your data
[status] Preparing your response
# User Output
There is no data available in the dbo.CompletedChecklist table to analyze. Therefore, no daily completion counts or charts can be generated.

# Developer Output
Data Quality Notes: The dbo.CompletedChecklist table contains no rows, so analysis could not proceed.
Technical Notes: Query to find the latest CreationDate and daily counts returned no data.
SQL Appendix:
-- Check for data presence
SELECT COUNT(*) AS total_rows FROM dbo.CompletedChecklist;

--- FINAL ---

# User Output
There is no data available in the dbo.CompletedChecklist table to analyze. Therefore, no daily completion counts or charts can be generated.

# Developer Output
Data Quality Notes: The dbo.CompletedChecklist table contains no rows, so analysis could not proceed.
Technical Notes: Query to find the latest CreationDate and daily counts returned no data.
SQL Appendix:
-- Check for data presence
SELECT COUNT(